In [ ]:
import scanpy as sc, anndata as ad
import pandas as pd
from glob import glob
import numpy as np
from tqdm import tqdm
import loompy as lp

In [ ]:
adata_sample_1 = sc.read_h5ad('../../data/dfci/expression-data_scrna_dfci_tumor_expr_CCG1106_075_T1.h5ad')
adata_sample_2 = sc.read_h5ad('../../data/dfci/expression-data_scrna_dfci_tumor_expr_CCG1106_084_T1.h5ad')
samples = [('CCG1106_075_T1', adata_sample_1), ('CCG1106_084_T1', adata_sample_2)]
hvg_all = []
for sample, sample_adata in samples:

    adata = sample_adata.copy()
    adata.X = adata.X.astype('int64')
    assert adata.X.dtype == 'int64'

    sc.pp.filter_genes(adata, min_cells = 1)
    print(adata.n_obs, adata.n_vars)
    adata.layers["counts"] = adata.X.copy()

    # log-normalize counts to TPM
    sc.pp.normalize_total(adata, target_sum=1e6)
    sc.pp.log1p(adata)
    print(adata.n_obs, adata.n_vars)

    loom_path = f'../data/{sample}_scenic_input.loom'
    
    row_attrs = {
        "Gene": np.array(adata.var_names)
    }
    col_attrs = {
        "CellID": np.array(adata.obs_names),
        "nGene": np.array(np.sum(adata.X.transpose() > 0, axis=0)).flatten(),
        "nUMI": np.array(np.sum(adata.X.transpose(), axis=0)).flatten(),
        "sample":np.array(adata.obs['sample']),
    }

    lp.create(loom_path, adata.X.transpose(), row_attrs, col_attrs)
    print(f"  Saved: {loom_path}")

1955 19813
1955 19813
  Saved: CCG1106_075_T1_scenic_input.loom
1273 19257
1273 19257
  Saved: CCG1106_084_T1_scenic_input.loom
